In [ ]:
#import libraries 
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

In [ ]:
#configure project path
PROJECT_ROOT = Path(r"E:\Alexandra PrJ\AI_Data_Analyst_Agent")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

print("Project root:", PROJECT_ROOT)
print("Environment file found:", ENV_PATH.exists())

In [ ]:
load_dotenv(ENV_PATH, override=True)

groq_api_key = os.getenv("GROQ_API_KEY")

print("Groq API key loaded:", groq_api_key is not None)

In [ ]:
%pip install langchain-groq

In [ ]:
#initialize LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=groq_api_key
)

print("LLM initialized successfully.")

In [ ]:
#testing llm response 
response = llm.invoke(
    "What is the role of SQL in data analysis? Answer in two sentences."
)

print(response.content)

In [ ]:
#import database tools 
from src.tools.inspect_schema import inspect_schema
from src.tools.query_database import query_database


In [ ]:
#inspect database schema 
database_schema = inspect_schema()

print(database_schema.keys())

In [ ]:
#format database schema 
def format_schema(schema):
    schema_text = ""

    for table_name, table_info in schema.items():
        schema_text += f"\nTable: {table_name}\n"
        schema_text += "Columns:\n"

        for column in table_info["columns"]:
            schema_text += (
                f"- {column['name']} "
                f"({column['type']}, nullable={column['nullable']})\n"
            )

        schema_text += (
            f"Primary key: {table_info['primary_key']}\n"
        )

        if table_info["foreign_keys"]:
            schema_text += "Foreign keys:\n"

            for foreign_key in table_info["foreign_keys"]:
                schema_text += (
                    f"- {foreign_key['columns']} -> "
                    f"{foreign_key['referenced_table']}"
                    f"{foreign_key['referenced_columns']}\n"
                )

    return schema_text


schema_text = format_schema(database_schema)

print(schema_text)

In [ ]:
#generating SQL with LLM 
user_question = "What are the top 5 product categories by revenue?"

prompt = f"""
You are an expert PostgreSQL data analyst.

Use only the database schema provided below.

Database schema:
{schema_text}

User question:
{user_question}

Rules:
- Generate valid PostgreSQL.
- Use only tables and columns from the schema.
- Do not invent columns or tables.
- Revenue is defined as the sum of order_items.price.
- Return only the SQL query.
- Do not use Markdown code blocks.
"""

response = llm.invoke(prompt)

generated_sql = response.content.strip()

print(generated_sql)

In [ ]:
#executed generated SQL
query_result = query_database(generated_sql)

print(query_result)

In [ ]:
#generating final answer
result_text = query_result.to_string(index=False)

answer_prompt = f"""
You are an AI data analyst.

User question:
{user_question}

SQL result:
{result_text}

Answer the user's question clearly and concisely.
Use only the information provided in the SQL result.
Do not invent additional facts.
"""

final_response = llm.invoke(answer_prompt)

print(final_response.content)

In [ ]:
#creating data analyst function 
def ask_data_analyst(question):
    sql_prompt = f"""
You are an expert PostgreSQL data analyst.

Use only the database schema provided below.

Database schema:
{schema_text}

User question:
{question}

Rules:
- Generate valid PostgreSQL.
- Use only tables and columns from the schema.
- Do not invent columns or tables.
- Revenue is defined as the sum of order_items.price.
- Return only the SQL query.
- Do not use Markdown code blocks.
"""

    sql_response = llm.invoke(sql_prompt)
    sql_query = sql_response.content.strip()

    result = query_database(sql_query)
    result_text = result.to_string(index=False)

    answer_prompt = f"""
You are an AI data analyst.

User question:
{question}

SQL result:
{result_text}

Answer the user's question clearly and concisely.
Use only the information provided in the SQL result.
Do not invent additional facts.
"""

    answer_response = llm.invoke(answer_prompt)

    return {
        "question": question,
        "sql": sql_query,
        "result": result,
        "answer": answer_response.content
    }

In [ ]:
#test
analysis = ask_data_analyst(
    "Which 5 customer states generated the highest revenue?"
)

print("Generated SQL:")
print(analysis["sql"])

print("\nQuery result:")
print(analysis["result"])

print("\nFinal answer:")
print(analysis["answer"])

In [ ]:
%pip install langgraph

In [ ]:
from langchain_core.tools import tool

#creating schema tools
@tool
def inspect_database_schema():
    """Inspect the PostgreSQL database schema, including tables, columns, primary keys, and foreign keys."""
    schema = inspect_schema()
    return format_schema(schema)
print(inspect_database_schema.invoke({}))

In [ ]:
#creating database query tool

@tool
def execute_database_query(query: str):
    """Execute a read-only PostgreSQL query and return the result or an error message."""
    try:
        result = query_database(query)

        if result.empty:
            return "The query returned no results."

        return result.to_string(index=False)

    except Exception as error:
        return f"SQL_ERROR: {str(error)}"

In [ ]:
result = execute_database_query.invoke(
    {
        "query": "SELECT COUNT(*) AS total_orders FROM orders;"
    }
)

print(result)

In [ ]:
tools = [
    inspect_database_schema,
    execute_database_query
]

llm_with_tools = llm.bind_tools(tools)

print("Tools successfully bound to LLM.")

In [ ]:
response = llm_with_tools.invoke(
    "How many orders are stored in the database? Use the available tools to answer."
)

print("Response:")
print(response.content)

print("\nTool calls:")
print(response.tool_calls)

In [ ]:
#import langGraph components

from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.messages import AnyMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

In [ ]:
#define state agent
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
#creating LLM node

from langchain_core.messages import SystemMessage

SYSTEM_PROMPT = """
You are an AI Data Analyst working with a PostgreSQL e-commerce database.

Use the available tools to inspect the database schema and execute read-only SQL queries.

Business rules:
- Revenue means product revenue and must be calculated as SUM(order_items.price).
- When calculating revenue, include all order_items by default.
- Do not filter by order_status unless the user explicitly asks for delivered, canceled, shipped, or another specific order status.
- Do not use payments.payment_value as revenue unless the user explicitly asks about payments or total paid amounts.
- Use only tables and columns that exist in the database schema.
- Inspect the database schema before writing SQL when necessary.
- Never invent tables, columns, relationships, or query results.
- Use PostgreSQL syntax.
- Prefer database results over assumptions.
- Clearly distinguish revenue from freight costs and payment values.
- Answer using only information obtained from the available tools.
- Do not claim causation from correlation or observational data.
- Do not invent business explanations that are not supported by query results.
- Clearly distinguish observed facts from hypotheses or possible explanations.
- Before interpreting unusually low values at the beginning or end of a time series, consider whether the period may be incomplete.
- If the available data is insufficient to explain why a trend or anomaly occurred, explicitly say that the cause cannot be determined from the current data.
- When analyzing a time series, always query the minimum and maximum timestamp of the relevant date column before interpreting the first or last period.
- You must verify whether boundary periods are complete using a database query before describing them as incomplete, partial, anomalous, or representative of a business decline.
- Never infer business maturity, market expansion, customer adoption, business decline, or similar business causes from a revenue trend alone.
- Describe observed numerical patterns separately from possible explanations.
- If a possible explanation cannot be verified with the available database data, explicitly state that the cause cannot be determined from the current data.
"""

def call_model(state: AgentState):
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"]
    ]

    response = llm_with_tools.invoke(messages)

    return {
        "messages": [response]
    }

In [ ]:
#define routing logic
def should_continue(state: AgentState):
    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END

In [ ]:
#build agent grapg

tool_node = ToolNode(tools)

graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue
)

graph_builder.add_edge("tools", "agent")

agent = graph_builder.compile()

print("Agent graph compiled successfully.")

In [ ]:
#test agent
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "How many orders are stored in the database?"
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

In [ ]:
#test complex
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which 5 customer states generated the highest revenue?"
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

In [ ]:
# Testing SQL error handling

error_result = execute_database_query.invoke(
    {
        "query": "SELECT nonexistent_column FROM orders;"
    }
)

print(error_result)

In [ ]:
#rebuild agent new tools 
tools = [
    inspect_database_schema,
    execute_database_query
]

llm_with_tools = llm.bind_tools(tools)

tool_node = ToolNode(tools)

graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue
)

graph_builder.add_edge("tools", "agent")

agent = graph_builder.compile()

print("Agent rebuilt successfully.")

In [ ]:
#test SQL self-correction

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
Find the total number of orders.

For this test, first call execute_database_query using:
SELECT COUNT(*) FROM nonexistent_orders;

After receiving the SQL error, correct the query using the database schema and return the correct answer.
"""
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

In [ ]:
#import Python Analysis
from src.tools.python_analysis import analyze_query_result

print("Python analysis function imported successfully.")

In [ ]:
#test Python analysis tool

analysis_result = analyze_query_result(
    query="""
        SELECT
            EXTRACT(
                EPOCH FROM (
                    order_delivered_customer_date - order_purchase_timestamp
                )
            ) / 86400 AS delivery_days
        FROM orders
        WHERE order_delivered_customer_date IS NOT NULL;
    """,
    operation="mean",
    column="delivery_days"
)

print(analysis_result)

In [ ]:
#load real data for Python Analysis
delivery_review_data = query_database("""
SELECT
    EXTRACT(
        EPOCH FROM (
            o.order_delivered_customer_date - o.order_purchase_timestamp
        )
    ) / 86400 AS delivery_days,
    r.review_score
FROM orders o
JOIN reviews r
    ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL;
""")

print(delivery_review_data.head())
print()
print("Rows:", len(delivery_review_data))

In [ ]:
#analyze database results with Python
analysis_result = analyze_query_result(
    query="""
        SELECT
            EXTRACT(
                EPOCH FROM (
                    o.order_delivered_customer_date - o.order_purchase_timestamp
                )
            ) / 86400 AS delivery_days,
            r.review_score
        FROM orders o
        JOIN reviews r
            ON o.order_id = r.order_id
        WHERE o.order_delivered_customer_date IS NOT NULL
          AND r.review_score IS NOT NULL;
    """,
    operation="correlation"
)

print(analysis_result)

In [ ]:
#import database analysis function 
import importlib
import src.tools.python_analysis as python_analysis

importlib.reload(python_analysis)

print(dir(python_analysis))

In [ ]:
from src.tools.python_analysis import analyze_query_result

print("Python database analysis imported successfully.")

In [ ]:
#creating Python analysis tool

from langchain_core.tools import tool


@tool("python")
def python(
    query: str,
    operation: str,
    column: str = None
):
    """Analyze PostgreSQL query results using Pandas for statistics and correlations."""

    try:
        if operation in ["corr", "cor"]:
            operation = "correlation"

        return analyze_query_result(
            query=query,
            operation=operation,
            column=column
        )

    except Exception as error:
        return f"PYTHON_ANALYSIS_ERROR: {str(error)}"


print("Python tool created successfully.")

In [ ]:
#recreate llm

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    max_tokens=2048,
    reasoning_effort="low",
    api_key=groq_api_key
)

print("LLM recreated successfully.")

In [ ]:
#update agent tools

tools = [
    inspect_database_schema,
    execute_database_query,
    python
]

llm_with_tools = llm.bind_tools(tools)

print("Tools rebound successfully.")

In [ ]:
#rebuild agent with Python Analysis
tool_node = ToolNode(tools)

graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue
)

graph_builder.add_edge("tools", "agent")

agent = graph_builder.compile()

print("Agent rebuilt successfully.")

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Is delivery time related to customer review scores? Calculate the correlation and explain the result."
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

In [ ]:
#import chart tool
from src.tools.create_chart import create_chart_from_query

print("Chart tool imported successfully.")

In [ ]:
#test chart generation 
chart_path = create_chart_from_query(
    query="""
        SELECT
            TO_CHAR(o.order_purchase_timestamp, 'YYYY-MM') AS month,
            SUM(oi.price) AS revenue
        FROM orders o
        JOIN order_items oi
            ON o.order_id = oi.order_id
        GROUP BY month
        ORDER BY month;
    """,
    chart_type="line",
    x_column="month",
    y_column="revenue",
    title="Monthly Revenue",
    filename="monthly_revenue.png"
)

print("Chart saved to:", chart_path)

In [ ]:
#create chart generation tool

@tool
def create_chart(
    query: str,
    chart_type: str,
    x_column: str,
    y_column: str,
    title: str = "Data Analysis",
    filename: str = "chart.png"
):
    """Create a chart from PostgreSQL data and save it as an image."""

    try:
        return create_chart_from_query(
            query=query,
            chart_type=chart_type,
            x_column=x_column,
            y_column=y_column,
            title=title,
            filename=filename
        )

    except Exception as error:
        return f"CHART_ERROR: {str(error)}"


print("Chart tool created successfully.")

In [ ]:
print(create_chart.name)
print(create_chart.description)

In [ ]:
#update agent tools

tools = [
    inspect_database_schema,
    execute_database_query,
    python,
    create_chart
]

llm_with_tools = llm.bind_tools(tools)

print("Four tools successfully bound to LLM.")

In [ ]:
#rebuild agent

tool_node = ToolNode(tools)

graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue
)

graph_builder.add_edge("tools", "agent")

agent = graph_builder.compile()

print("Agent rebuilt with chart tool successfully.")

In [ ]:
#test
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Show me the monthly revenue trend as a line chart."
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

In [ ]:
#testing revenue bussines rules

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the total revenue?"
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)

In [ ]:
#importing report export tool
from src.tools.export_report import export_analysis_report

print("Report export function imported successfully.")

In [ ]:
#test report export

report_path = export_analysis_report(
    title="Revenue Analysis",
    question="What is the total revenue?",
    analysis="The total product revenue is 13,591,643.70.",
    filename="revenue_analysis.md"
)

print("Report saved to:", report_path)

In [ ]:
#creating report export tool

@tool
def export_report(
    title: str,
    question: str,
    analysis: str,
    chart_path: str = None,
    filename: str = "analysis_report.md"
):
    """Export an analysis result as a Markdown report, optionally including a chart."""

    try:
        return export_analysis_report(
            title=title,
            question=question,
            analysis=analysis,
            chart_path=chart_path,
            filename=filename
        )

    except Exception as error:
        return f"REPORT_ERROR: {str(error)}"


print("Report export tool created successfully.")

In [ ]:
print(export_report.name)
print(export_report.description)

In [ ]:
#update agent tools

tools = [
    inspect_database_schema,
    execute_database_query,
    python,
    create_chart,
    export_report
]

llm_with_tools = llm.bind_tools(tools)

print("Five tools successfully bound to LLM.")

In [ ]:
#rebuild agent

tool_node = ToolNode(tools)

graph_builder = StateGraph(AgentState)

graph_builder.add_node("agent", call_model)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "agent")

graph_builder.add_conditional_edges(
    "agent",
    should_continue
)

graph_builder.add_edge("tools", "agent")

agent = graph_builder.compile()

print("Agent rebuilt with five tools successfully.")

In [ ]:
# Test complete analysis

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Analyze the monthly revenue trend. "
                    "Create a line chart and export the analysis as a Markdown report."
                )
            }
        ]
    }
)

for message in result["messages"]:
    print(type(message).__name__)
    print(message.content)

    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:", message.tool_calls)

    print("-" * 50)